# Project FORESIGHT: AI-Powered Demand & Inventory Intelligence
End-to-end notebook: **data → EDA → features → models → backtest → 30-day forecast → inventory recommendations**.

**How to run (Google Colab):** Runtime → Run all. When asked, upload `sales_daily.csv`.
The last cells write `forecasts.csv`, `recommendations.csv`, etc. to the `outputs/` folder for the Streamlit dashboard.

> Stock levels, lead times and unit cost are **simulated** (the dataset has no inventory data). The 30-day forecast assumes no promotions.

In [ ]:
!pip install -q lightgbm plotly pandas numpy scikit-learn

## 1. Load data

In [ ]:
import os, json, numpy as np, pandas as pd, plotly.express as px
os.makedirs("data", exist_ok=True); os.makedirs("outputs", exist_ok=True)
if not os.path.exists("data/sales_daily.csv"):
    try:
        from google.colab import files
        up = files.upload()                       # choose sales_daily.csv
        os.replace(list(up.keys())[0], "data/sales_daily.csv")
    except ImportError:
        raise FileNotFoundError("Put sales_daily.csv inside a data/ folder next to this notebook.")
raw = pd.read_csv("data/sales_daily.csv", parse_dates=["Date"])
print(raw.shape, raw.Date.min().date(), "->", raw.Date.max().date(), "| SKUs:", raw.SKU.nunique())
print("Missing:", int(raw.isna().sum().sum()), "| Duplicates:", int(raw.duplicated(["Date","SKU"]).sum()))
raw.head()

## 2. Exploratory analysis

In [ ]:
tot = raw.groupby("Date")["Units_Sold"].sum().reset_index()
px.line(tot, x="Date", y="Units_Sold", title="Total daily units sold").show()
dow = tot.assign(day=tot.Date.dt.day_name()).groupby("day")["Units_Sold"].mean() \
         .reindex(["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]).reset_index()
px.bar(dow, x="day", y="Units_Sold", title="Average units by weekday").show()
mo = tot.groupby(tot.Date.dt.month)["Units_Sold"].mean().reset_index()
px.bar(mo, x="Date", y="Units_Sold", title="Average units by month").update_xaxes(title="Month").show()
lift = raw.groupby("Promotion")["Units_Sold"].mean()
print(f"Promotion lift: +{(lift[1]/lift[0]-1)*100:.0f}% units on promo days")

## 3. Feature engineering
Lags and rolling statistics are shifted so a day only sees the past (no leakage).

In [ ]:
"""Feature engineering shared by training (vectorised) and recursive forecasting."""
import numpy as np
import pandas as pd

FEATURES = [
    "sku_code", "dow", "month", "doy", "is_weekend", "promo", "price",
    "lag_7", "lag_14", "lag_28", "roll_mean_7", "roll_mean_28", "roll_std_28",
]


def load_sales(path):
    df = pd.read_csv(path, parse_dates=["Date"])
    df = df.sort_values(["SKU", "Date"]).reset_index(drop=True)
    df["sku_code"] = df["SKU"].astype("category").cat.codes
    return df


def build_training_frame(df):
    """Lags/rolling stats are shifted so day t only sees data up to t-7 (lags) or t-1 (rolling)."""
    d = df.copy()
    g = d.groupby("SKU")["Units_Sold"]
    d["lag_7"], d["lag_14"], d["lag_28"] = g.shift(7), g.shift(14), g.shift(28)
    s1 = g.shift(1)
    grp = s1.groupby(d["SKU"])
    d["roll_mean_7"] = grp.transform(lambda x: x.rolling(7).mean())
    d["roll_mean_28"] = grp.transform(lambda x: x.rolling(28).mean())
    d["roll_std_28"] = grp.transform(lambda x: x.rolling(28).std())
    d["dow"] = d["Date"].dt.dayofweek
    d["month"] = d["Date"].dt.month
    d["doy"] = d["Date"].dt.dayofyear
    d["is_weekend"] = (d["dow"] >= 5).astype(int)
    d["promo"] = d["Promotion"]
    d["price"] = d["Price"]
    return d


def step_features(hist, date, promo, price, sku_codes):
    """Features for ONE future day for all SKUs. hist: (n_sku, T) units up to the day before `date`."""
    dow = date.dayofweek
    return pd.DataFrame({
        "sku_code": sku_codes,
        "dow": dow, "month": date.month, "doy": date.dayofyear,
        "is_weekend": int(dow >= 5),
        "promo": promo, "price": price,
        "lag_7": hist[:, -7], "lag_14": hist[:, -14], "lag_28": hist[:, -28],
        "roll_mean_7": hist[:, -7:].mean(axis=1),
        "roll_mean_28": hist[:, -28:].mean(axis=1),
        "roll_std_28": hist[:, -28:].std(axis=1, ddof=1),
    })[FEATURES]

## 4. Models, recursive forecasting and backtest
Baselines: naive (same weekday last week) and 28-day moving average. Main model: one global LightGBM.

In [ ]:
"""Model training, honest multi-step backtesting and recursive forecasting."""
import numpy as np
import pandas as pd
import lightgbm as lgb

PARAMS = dict(objective="regression", n_estimators=600, learning_rate=0.03, num_leaves=31,
              min_child_samples=30, subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
              random_state=42, verbose=-1)


def train_lgbm(train_df):
    tr = train_df.dropna(subset=FEATURES)
    model = lgb.LGBMRegressor(**PARAMS)
    model.fit(tr[FEATURES], tr["Units_Sold"], categorical_feature=["sku_code"])
    return model


def recursive_forecast(model, hist, start_date, promo_plan, price, sku_codes):
    """Forecast promo_plan.shape[1] days ahead, feeding predictions back in as history."""
    horizon = promo_plan.shape[1]
    h = hist.astype(float).copy()
    preds = np.zeros((h.shape[0], horizon))
    for k in range(horizon):
        date = start_date + pd.Timedelta(days=k)
        X = step_features(h, date, promo_plan[:, k], price, sku_codes)
        p = np.clip(model.predict(X), 0, None)
        preds[:, k] = p
        h = np.concatenate([h, p[:, None]], axis=1)
    return preds


def wape(y, p): return float(np.abs(y - p).sum() / y.sum() * 100)
def mae(y, p): return float(np.abs(y - p).mean())
def rmse(y, p): return float(np.sqrt(((y - p) ** 2).mean()))
def bias(y, p): return float((p - y).sum() / y.sum() * 100)


def backtest(model, U, P, price, dates, sku_codes, test_days=90, horizon=30):
    """Rolling-origin backtest: forecast `horizon` days at a time, starting from actual history.
    Mirrors how the model is used in production (30 days ahead, no peeking)."""
    T = U.shape[1]
    t0 = T - test_days
    lgbm = np.zeros((U.shape[0], test_days))
    naive = np.zeros_like(lgbm)   # same weekday, last observed week
    ma28 = np.zeros_like(lgbm)    # flat 28-day moving average
    for w in range(test_days // horizon):
        o = t0 + w * horizon
        hist = U[:, :o]
        sl = slice(w * horizon, (w + 1) * horizon)
        lgbm[:, sl] = recursive_forecast(model, hist, dates[o], P[:, o:o + horizon], price, sku_codes)
        for k in range(horizon):
            naive[:, w * horizon + k] = hist[:, -7:][:, k % 7]
        ma28[:, sl] = hist[:, -28:].mean(axis=1, keepdims=True)
    return U[:, t0:], {"Naive (last week)": naive, "Moving average (28d)": ma28, "LightGBM": lgbm}

## 5. Inventory intelligence
Safety stock, reorder point, EOQ, days of cover, risk flags and order quantities.

In [ ]:
"""Inventory intelligence: safety stock, reorder point, EOQ, risk flags, order recommendations."""
import numpy as np
import pandas as pd

Z = {0.90: 1.2816, 0.95: 1.6449, 0.97: 1.8808, 0.99: 2.3263}
REVIEW_DAYS = 7          # how often stock is reviewed / ordered
OVERSTOCK_COVER = 60     # days of cover above which stock is "excess"
COST_RATIO = 0.65        # simulated unit cost as share of selling price
ORDERING_COST = 2000     # simulated fixed cost per purchase order (same currency as Price)
HOLDING_RATE = 0.20      # annual holding cost as share of unit cost


def simulate_inventory(sku_summary, seed=42):
    """The dataset has no stock data, so these columns are SIMULATED (fixed seed, documented in README)."""
    rng = np.random.default_rng(seed)
    n = len(sku_summary)
    out = sku_summary.copy()
    out["lead_time_days"] = rng.integers(3, 15, n)
    # current cover drawn from a wide range so the demo shows healthy, low and excess stock
    cover = np.where(rng.random(n) < 0.35, rng.uniform(2, 14, n),
             np.where(rng.random(n) < 0.35, rng.uniform(60, 110, n), rng.uniform(15, 55, n)))
    out["current_stock"] = np.round(cover * out["avg_daily_demand_hist"]).astype(int)
    out["unit_cost"] = (out["price"] * COST_RATIO).round(2)
    return out


def compute_recommendations(df, service_level=0.95):
    """df needs: sku, avg_daily_demand (forecast), demand_std (daily forecast error),
    lead_time_days, current_stock, unit_cost, price."""
    z = Z[service_level]
    r = df.copy()
    d, L = r["avg_daily_demand"], r["lead_time_days"]
    r["safety_stock"] = np.ceil(z * r["demand_std"] * np.sqrt(L))
    r["reorder_point"] = np.ceil(d * L + r["safety_stock"])
    r["days_of_cover"] = (r["current_stock"] / d).round(1)
    annual = d * 365
    holding = r["unit_cost"] * HOLDING_RATE
    r["eoq"] = np.ceil(np.sqrt(2 * annual * ORDERING_COST / holding))
    target = d * (L + REVIEW_DAYS) + r["safety_stock"]
    r["recommended_order_qty"] = np.ceil(np.maximum(0, target - r["current_stock"]))

    stockout = (r["current_stock"] < r["reorder_point"])
    overstock = r["days_of_cover"] > OVERSTOCK_COVER
    r["flag"] = np.where(stockout, "Stockout risk", np.where(overstock, "Overstock", "Healthy"))
    r["priority"] = np.select(
        [stockout & (r["days_of_cover"] < L), stockout, overstock],
        ["Critical", "High", "Low"], default="-")

    # money impact
    r["expected_lost_sales_value"] = (np.maximum(0, d * L - r["current_stock"]) * r["price"]).round(0)
    r["excess_units"] = np.maximum(0, r["current_stock"] - d * OVERSTOCK_COVER).round(0)
    r["excess_inventory_value"] = (r["excess_units"] * r["unit_cost"]).round(0)
    r["order_value"] = (r["recommended_order_qty"] * r["unit_cost"]).round(0)
    r["action"] = np.select(
        [r["flag"] == "Stockout risk", r["flag"] == "Overstock"],
        ["Order " + r["recommended_order_qty"].astype(int).astype(str) + " units now",
         "Pause reordering; " + r["excess_units"].astype(int).astype(str) + " excess units"],
        default="No action")
    return r

## 6. Run the pipeline

In [ ]:
TEST_DAYS, HORIZON, HIST_DAYS_SHOWN = 90, 30, 120
df = load_sales("data/sales_daily.csv")
skus = sorted(df["SKU"].unique())
sku_codes = np.arange(len(skus))
piv = lambda c: df.pivot(index="SKU", columns="Date", values=c).loc[skus]
U, P = piv("Units_Sold").values.astype(float), piv("Promotion").values.astype(float)
dates = pd.DatetimeIndex(piv("Units_Sold").columns)
price = df.groupby("SKU")["Price"].last().loc[skus].values
T = U.shape[1]

# ---- 1. train on everything before the test window, backtest 30 days at a time ------
cut = dates[T - TEST_DAYS]
feat = build_training_frame(df)
model_bt = train_lgbm(feat[feat["Date"] < cut])
actual, preds = backtest(model_bt, U, P, price, dates, sku_codes, TEST_DAYS, HORIZON)

rows = []
for name, p in preds.items():
    rows.append(dict(model=name, MAE=mae(actual, p), RMSE=rmse(actual, p), WAPE_pct=wape(actual, p), Bias_pct=bias(actual, p)))
metrics = pd.DataFrame(rows).round(3)
best_naive = metrics.loc[metrics.model == "Naive (last week)", "WAPE_pct"].iloc[0]
lgb_w = metrics.loc[metrics.model == "LightGBM", "WAPE_pct"].iloc[0]
print(metrics.to_string(index=False))
print(f"LightGBM improves WAPE vs naive by {(best_naive - lgb_w) / best_naive * 100:.1f}%")
metrics.to_csv("outputs/model_metrics.csv", index=False)

# per-SKU accuracy + error std (used for safety stock)
lg = preds["LightGBM"]
err_std = (actual - lg).std(axis=1, ddof=1)
sku_acc = pd.DataFrame({"sku": skus, "wape_pct": [wape(actual[i], lg[i]) for i in range(len(skus))],
                        "demand_std": err_std}).round(3)

# ---- 2. final model on ALL data, 30-day forward forecast ------------------------------
model = train_lgbm(feat)
fut_start = dates[-1] + pd.Timedelta(days=1)
promo_plan = np.zeros((len(skus), HORIZON))          # assumption: no promotions planned
fut = recursive_forecast(model, U, fut_start, promo_plan, price, sku_codes)
fut_dates = pd.date_range(fut_start, periods=HORIZON)

imp = pd.DataFrame({"feature": FEATURES, "importance": model.booster_.feature_importance("gain")})
imp["importance"] = (imp["importance"] / imp["importance"].sum() * 100).round(2)
imp.sort_values("importance", ascending=False).to_csv("outputs/feature_importance.csv", index=False)

# ---- 3. forecasts.csv for the dashboard --------------------------------------------------
Z95 = 1.6449
parts = []
for i, s in enumerate(skus):
    h = slice(T - TEST_DAYS - (HIST_DAYS_SHOWN - TEST_DAYS), T - TEST_DAYS)
    parts.append(pd.DataFrame({"date": dates[h], "sku": s, "actual": U[i, h], "forecast": np.nan, "type": "history"}))
    parts.append(pd.DataFrame({"date": dates[T - TEST_DAYS:], "sku": s, "actual": actual[i], "forecast": lg[i], "type": "backtest"}))
    parts.append(pd.DataFrame({"date": fut_dates, "sku": s, "actual": np.nan, "forecast": fut[i], "type": "future"}))
fc = pd.concat(parts, ignore_index=True)
fc["lower"] = (fc["forecast"] - Z95 * err_std[fc["sku"].map({s: i for i, s in enumerate(skus)}).values]).clip(lower=0)
fc["upper"] = fc["forecast"] + Z95 * err_std[fc["sku"].map({s: i for i, s in enumerate(skus)}).values]
fc.assign(date=fc['date'].dt.strftime('%Y-%m-%d')).round(2).to_csv("outputs/forecasts.csv", index=False)

# ---- 4. inventory recommendations ---------------------------------------------------------
summary = pd.DataFrame({
    "sku": skus,
    "price": price,
    "avg_daily_demand_hist": U[:, -28:].mean(axis=1),
    "avg_daily_demand": fut.mean(axis=1),
    "forecast_30d_units": fut.sum(axis=1),
    "hist_revenue": df.groupby("SKU")["Revenue"].sum().loc[skus].values,
}).merge(sku_acc, on="sku")
summary = simulate_inventory(summary, seed=42)
# ABC class by revenue share
summary = summary.sort_values("hist_revenue", ascending=False)
cum = summary["hist_revenue"].cumsum() / summary["hist_revenue"].sum()
summary["abc_class"] = np.where(cum <= 0.8, "A", np.where(cum <= 0.95, "B", "C"))
summary = summary.sort_values("sku").reset_index(drop=True)
summary.to_csv("outputs/sku_inputs.csv", index=False)      # dashboard recomputes with a service-level slider
rec = compute_recommendations(summary, 0.95)
rec.round(2).to_csv("outputs/recommendations.csv", index=False)

print(rec["flag"].value_counts().to_string())
print("Excess inventory value:", int(rec.excess_inventory_value.sum()),
      "| Lost-sales exposure:", int(rec.expected_lost_sales_value.sum()))
json.dump({"test_start": str(cut.date()), "test_end": str(dates[-1].date()), "forecast_start": str(fut_start.date()),
           "forecast_end": str(fut_dates[-1].date()), "wape_lightgbm": lgb_w, "wape_naive": best_naive}, open("outputs/run_info.json", "w"))

## 7. Results

In [ ]:
display(metrics)
px.bar(metrics, x="model", y="WAPE_pct", title="Backtest WAPE % (lower is better)", text="WAPE_pct").show()
px.bar(imp.sort_values("importance"), x="importance", y="feature", orientation="h", title="Feature importance (gain %)").show()
rec[rec.flag=="Stockout risk"].sort_values("days_of_cover")[["sku","current_stock","reorder_point","days_of_cover","lead_time_days","recommended_order_qty","priority","action"]].head(10)

In [ ]:
sku = "SKU001"
d = fc[fc.sku == sku]
import plotly.graph_objects as go
f = go.Figure()
h = d[d.type.isin(["history","backtest"])]
f.add_scatter(x=h.date, y=h.actual, name="Actual")
b = d[d.type=="backtest"]; f.add_scatter(x=b.date, y=b.forecast, name="Backtest forecast", line=dict(dash="dot"))
u = d[d.type=="future"]; f.add_scatter(x=u.date, y=u.forecast, name="30-day forecast")
f.add_scatter(x=u.date, y=u.upper, line=dict(width=0), showlegend=False)
f.add_scatter(x=u.date, y=u.lower, fill="tonexty", line=dict(width=0), name="95% interval")
f.update_layout(title=f"{sku}: actual vs forecast"); f.show()

In [ ]:
# Download the outputs (Colab)
try:
    from google.colab import files
    import shutil; shutil.make_archive("foresight_outputs", "zip", "outputs"); files.download("foresight_outputs.zip")
except ImportError:
    print("Outputs saved in ./outputs")